# Notebook 05 — The Downstream Economics of Arbitrage-Free Surfaces

NB02–NB04 built and audited three families of smoothers (SVI/SSVI, per-day
derivative-constrained deep smoother, neural operators) and established the thesis's
methodological result: **soft no-arbitrage constraints control derivatives of the surface,
and can be numerically satisfied while economically void** (the collocation blind spot; the
across-days transfer gap of the operator). This notebook closes the argument by pricing
with those derivatives:

| § | Object | Why it is the constraints made visible |
|---|---|---|
| **A** | Dupire local volatility | $\sigma^2_{loc} = \partial_\tau w / g$ — calendar is the numerator, butterfly the denominator; a violation is a negative/exploding local variance that must be *repaired* before one path can be drawn. |
| **C** | Risk-neutral densities | $q(k) = g\,\varphi(d_2)/\sqrt{w}$: positivity ⇔ butterfly; negative mass is the violation in probability units. |
| **E** | Residual economics | IC + cost-aware decile spread of (quote − surface) residuals on held-out quotes; clean λ=10 vs dirty λ=0. *Not* an alpha claim. |
| **F** | Trading the violations | F1: executable arbitrage scan at bid/ask. F2: residual relative-value backtest on a common universe (horizon x execution grid), fit-gated, cost-aware. |
| **G** | Detection power | Synthetic mispricing (amplitude a, half-life h) injected into REAL quotes with REAL spreads and REAL noise: the net-profitability frontier of this market, vs its empirical residual distribution. |
| **D** | VIX replication | 30-day variance-swap rate from each surface vs the published VIX: external ground truth for the wings. Gated on `vix_daily.csv`. |

**Inputs** (all optional — sections skip gracefully): `option_prices_clean.parquet` (NB01);
`nb03_surfaces/*.npz` (exact autodiff fields; λ=0 pack = dirty counterfactual);
`nb04_surfaces/*.npz` (w only; FD fields validated below); SVI/SSVI refit here per day.

**Holdout protocol.** NB05 draws its own 20% per-day holdout (`crc32(date)+1`), applied
identically to every family.

In [ ]:
# ## 0. Config, loaders, Black machinery
import os, glob, zlib, time, re
from datetime import date as _date, datetime as _datetime
from pathlib import Path

import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize
from scipy.stats import spearmanr
from scipy.interpolate import RegularGridInterpolator

OUT_DIR   = Path(os.environ.get("THESIS_OUT_DIR", "data/clean"))
REAL_PARQUET = Path(os.environ.get("THESIS_OPT_PARQUET", str(OUT_DIR / "option_prices_clean.parquet")))
NB03_SURF = OUT_DIR / "nb03_surfaces"
NB04_SURF = OUT_DIR / "nb04_surfaces"
VIX_CSV   = Path(os.environ.get("THESIS_VIX_CSV", str(OUT_DIR / "vix_daily.csv")))   # date,vix
VIX_RAW   = Path(os.environ.get("THESIS_VIX_RAW", "data/raw/VIX_History.csv"))       # CBOE file

SEED           = 0
HOLDOUT_FRAC   = 0.20
VIOL_TOL_MAT   = 1e-3     # same materiality threshold as NB03/NB04
MIN_PTS_SLICE  = 6
V_LOC_FLOOR    = 1e-4     # repaired local-variance bounds (report the repair rate, always)
V_LOC_CAP      = 4.0
MC_PATHS       = 20_000
MC_STEPS       = 120
FLAT_COST_VP   = 0.5      # fallback half-spread (vol points) when quote spreads are unavailable

rng = np.random.default_rng(SEED)
print(f"NB03 packs: {NB03_SURF.exists()} | NB04 packs: {NB04_SURF.exists()} | "
      f"quotes: {REAL_PARQUET.exists()} | VIX csv: {VIX_CSV.exists()}")


def holdout_mask(date, n):
    """NB05's uniform holdout: same rng for every family on a given day."""
    r = np.random.default_rng(zlib.crc32(str(date).encode()) + 1)
    m = r.random(n) < HOLDOUT_FRAC
    if m.all() or (~m).sum() < 20:
        m[:] = False
    return m


# ---------- normalized (undiscounted, forward) Black machinery ----------
from math import erf, sqrt as _sqrt


def _Phi(x):
    return 0.5 * (1.0 + np.vectorize(erf)(np.asarray(x, float) / np.sqrt(2.0)))


def _phi(x):
    return np.exp(-0.5 * np.asarray(x, float) ** 2) / np.sqrt(2 * np.pi)


def black_call(k, w):
    """Normalized undiscounted call C/F with total variance w at log-moneyness k = log(K/F)."""
    w = np.maximum(np.asarray(w, float), 1e-12)
    sw = np.sqrt(w)
    d1 = (-np.asarray(k, float) + w / 2) / sw
    return _Phi(d1) - np.exp(k) * _Phi(d1 - sw)


def black_put(k, w):
    return black_call(k, w) - 1.0 + np.exp(k)      # parity: c - p = 1 - e^k


def black_vega_w(k, w):
    """d(price)/d(sqrt-of-w), used only for weighting."""
    w = np.maximum(np.asarray(w, float), 1e-12)
    return _phi((-np.asarray(k, float) + w / 2) / np.sqrt(w))


def implied_w(price, k, is_call, lo=1e-10, hi=25.0, iters=80):
    """Vectorized bisection for total variance from a normalized price. Monotone in w."""
    price = np.asarray(price, float); k = np.asarray(k, float)
    lo = np.full_like(price, lo); hi = np.full_like(price, hi)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        pm = np.where(is_call, black_call(k, mid), black_put(k, mid))
        too_low = pm < price
        lo = np.where(too_low, mid, lo)
        hi = np.where(too_low, hi, mid)
    return 0.5 * (lo + hi)


_k = np.array([-0.3, 0.0, 0.2]); _w = np.array([0.04, 0.02, 0.06])
assert np.max(np.abs(implied_w(black_call(_k, _w), _k, np.array([True] * 3)) - _w)) < 1e-7
print("Black round-trip self-test OK")

In [ ]:
# === THESIS_HEADLESS_BOOTSTRAP_V1 — inserted for headless cluster runs ===
# Makes the notebook safe to execute with `jupyter nbconvert --execute` on a
# machine with no display, and exports every plotly figure to disk so the
# thesis LaTeX can \includegraphics them directly.
import os as _os
from pathlib import Path as _Path

import plotly.io as _pio
import plotly.graph_objects as _go

_HEADLESS = _os.environ.get("THESIS_HEADLESS", "0") == "1"

# figure output directory: defaults to <OUT_DIR>/figures, overridable
try:
    _fig_base = OUT_DIR
except NameError:
    _fig_base = _Path("data/clean")
FIG_DIR = _Path(_os.environ.get("THESIS_FIG_DIR", str(_Path(_fig_base) / "figures")))
FIG_DIR.mkdir(parents=True, exist_ok=True)

FIG_PREFIX = _os.environ.get("THESIS_FIG_PREFIX", "nb05")
FIG_SHARD = _os.environ.get("NB03_SHARD", "")
FIG_FORMAT = _os.environ.get("THESIS_FIG_FORMAT", "pdf")   # pdf = vector, best for LaTeX
FIG_WRITE_HTML = _os.environ.get("THESIS_FIG_HTML", "1") == "1"

if _HEADLESS:
    # never try to open a browser / require a live kernel front-end
    _pio.renderers.default = "json"
else:
    _pio.renderers.default = _os.environ.get("PLOTLY_RENDERER", "notebook_connected")

_FIG_COUNTER = {"n": 0}
_FIG_MANIFEST = []
_orig_fig_show = _go.Figure.show


def _thesis_fig_show(self, *args, **kwargs):
    """Save the figure, then display it only when running interactively."""
    _FIG_COUNTER["n"] += 1
    suffix = f"_{FIG_SHARD}" if FIG_SHARD else ""
    stem = f"{FIG_PREFIX}{suffix}_fig{_FIG_COUNTER['n']:02d}"

    saved = []
    try:
        path = FIG_DIR / f"{stem}.{FIG_FORMAT}"
        self.write_image(str(path), width=1100, height=700, scale=2)
        saved.append(path.name)
    except Exception as exc:                       # kaleido missing / export failure
        print(f"[fig] static export failed for {stem} "
              f"({exc.__class__.__name__}: {exc}) — falling back to HTML")

    if FIG_WRITE_HTML:
        try:
            path = FIG_DIR / f"{stem}.html"
            self.write_html(str(path), include_plotlyjs="cdn")
            saved.append(path.name)
        except Exception as exc:
            print(f"[fig] html export failed for {stem}: {exc}")

    _FIG_MANIFEST.append(dict(stem=stem, files=saved,
                              title=str(getattr(self.layout.title, "text", "") or "")))
    if saved:
        print(f"[fig] {stem} -> {', '.join(saved)}")

    if not _HEADLESS:
        return _orig_fig_show(self, *args, **kwargs)
    return None


_go.Figure.show = _thesis_fig_show


def write_figure_manifest():
    """Write the figure index so the LaTeX side knows what was produced."""
    try:
        import polars as _plr
        if _FIG_MANIFEST:
            rows = [dict(stem=r["stem"], files=";".join(r["files"]), title=r["title"])
                    for r in _FIG_MANIFEST]
            suffix = f"_{FIG_SHARD}" if FIG_SHARD else ""
            out = FIG_DIR / f"{FIG_PREFIX}{suffix}_figure_manifest.parquet"
            _plr.DataFrame(rows).write_parquet(out)
            print("written:", out)
    except Exception as exc:
        print("[fig] manifest not written:", exc)


print(f"[bootstrap] headless={_HEADLESS} | figures -> {FIG_DIR} "
      f"(.{FIG_FORMAT}) | prefix={FIG_PREFIX}{'_' + FIG_SHARD if FIG_SHARD else ''}")


## 0b. Surface packs and parametric baselines

A `SurfacePack` exposes `w(k,τ)`, `wt(k,τ)`, `g(k,τ)` by bilinear interpolation on the
exported fine grid. NB03 packs carry **exact autodiff** `wt`/`g`; NB04 packs carry `w` only,
so those fields are built by central finite differences on the fine export grid, validated
against the analytic SSVI baseline below (max error must sit far below `VIOL_TOL_MAT`).

In [ ]:
def durrleman_g(k, w, wk, wkk):
    return (1 - k * wk / (2 * w)) ** 2 - (wk ** 2 / 4) * (1 / w + 0.25) + wkk / 2


def fd_fields(kg, tg, W):
    """wt and g by central differences with COORDINATE-AWARE spacing (np.gradient with the
    coordinate arrays): supports the hybrid (exp U uniform) tau export grid, whose short-end
    density keeps the d_tau w reconstruction error ~3e-5 -- same lesson as the collocation
    chapter: derivative accuracy needs nodes where curvature lives."""
    Wk = np.gradient(W, kg, axis=1)
    Wkk = np.gradient(Wk, kg, axis=1)
    Wt = np.gradient(W, tg, axis=0)
    G = durrleman_g(kg[None, :], np.maximum(W, 1e-12), Wk, Wkk)
    return Wt, G


def tau_hybrid(t_lo, t_hi, nt=61):
    """exp U uniform tau grid (NB03's collocation design, reused as the export grid)."""
    te = np.exp(np.linspace(np.log(max(t_lo, 1e-4)), np.log(t_hi), nt))
    tu = np.linspace(t_lo, t_hi, nt)
    return np.unique(np.round(np.concatenate([te, tu]), 10))


class SurfacePack:
    def __init__(self, kg, tg, W, Wt=None, G=None, model="?", tag="", date=""):
        self.k, self.t, self.W = np.asarray(kg, float), np.asarray(tg, float), np.asarray(W, float)
        if Wt is None or G is None:
            Wt, G = fd_fields(self.k, self.t, self.W)
            self.exact = False
        else:
            self.exact = True
        self.Wt, self.G = np.asarray(Wt, float), np.asarray(G, float)
        self.model, self.tag, self.date = model, tag, date
        self._iw  = RegularGridInterpolator((self.t, self.k), self.W,  bounds_error=False, fill_value=None)
        self._iwt = RegularGridInterpolator((self.t, self.k), self.Wt, bounds_error=False, fill_value=None)
        self._ig  = RegularGridInterpolator((self.t, self.k), self.G,  bounds_error=False, fill_value=None)

    def w(self, k, tau):
        return np.maximum(self._iw(np.stack([np.clip(tau, self.t[0], self.t[-1]),
                                             np.clip(k, self.k[0], self.k[-1])], -1)), 1e-12)

    def wt(self, k, tau):
        return self._iwt(np.stack([np.clip(tau, self.t[0], self.t[-1]),
                                   np.clip(k, self.k[0], self.k[-1])], -1))

    def g(self, k, tau):
        return self._ig(np.stack([np.clip(tau, self.t[0], self.t[-1]),
                                  np.clip(k, self.k[0], self.k[-1])], -1))

    def iv(self, k, tau):
        return np.sqrt(self.w(k, tau) / np.maximum(tau, 1e-9))


def _scalar_value(x):
    a = np.asarray(x)
    return a.reshape(-1)[0] if a.size == 1 else x


def _scalar_text(x):
    x = _scalar_value(x)
    if isinstance(x, (bytes, np.bytes_)):
        x = x.decode("utf-8", errors="replace")
    return str(x).strip()


def canonical_date(x):
    """Return YYYY-MM-DD for numpy, Python, ISO, YYYYMMDD, or timestamp-like dates."""
    x = _scalar_value(x)
    if isinstance(x, (bytes, np.bytes_)):
        x = x.decode("utf-8", errors="replace")
    if isinstance(x, np.datetime64):
        return np.datetime_as_string(x, unit="D")
    if isinstance(x, (_datetime, _date)):
        return x.strftime("%Y-%m-%d")
    s = str(x).strip().strip("\"'")
    if (s.startswith("b'") and s.endswith("'")) or (s.startswith('b"') and s.endswith('"')):
        s = s[2:-1]
    m = re.search(r"(\d{4})[-/]?(\d{2})[-/]?(\d{2})", s)
    if not m:
        raise ValueError(f"Unrecognized date value: {x!r}")
    return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"


def option_key(exdate, strike):
    """Stable cross-day option key; avoids datetime formatting and float-noise mismatches."""
    return canonical_date(exdate), round(float(strike), 8)


def load_npz_packs(folder, model_filter=None):
    packs = {}
    for fn in sorted(glob.glob(str(folder / "*.npz"))):
        z = np.load(fn, allow_pickle=True)
        model = _scalar_text(z["model"]); tag = _scalar_text(z["tag"]); date = canonical_date(z["date"])
        if model_filter and model_filter not in model:
            continue
        key = (model if tag in ("", "lam10") else f"{model}_{tag}", date)
        packs[key] = SurfacePack(z["k"], z["tau"], z["w"],
                                 z["wt"] if "wt" in z.files else None,
                                 z["g"] if "g" in z.files else None,
                                 model=key[0], tag=tag, date=date)
    return packs


packs = {}
if NB03_SURF.exists():
    packs.update(load_npz_packs(NB03_SURF))
if NB04_SURF.exists():
    packs.update(load_npz_packs(NB04_SURF))
DATES = sorted({d for (_, d) in packs})
MODELS = sorted({m for (m, _) in packs})
print(f"loaded {len(packs)} pack(s) | models: {MODELS} | dates: {DATES}")

# ---------- SSVI baseline: GJ-certified prior refit here per day ----------
GJ_MARGIN = 3.8


def ssvi_w(k, theta, rho, eta, gamma):
    phi = eta * theta ** (-gamma)
    return 0.5 * theta * (1 + rho * phi * k + np.sqrt((phi * k + rho) ** 2 + (1 - rho ** 2)))


def gj_ok(rho, eta, gamma, alpha, beta, t_lo, t_hi):
    th_max = alpha * t_hi ** beta; th_min = alpha * max(t_lo, 1e-6) ** beta
    c1 = eta * th_max ** (1 - gamma) * (1 + abs(rho))
    c2 = eta ** 2 * max(th_max ** (1 - 2 * gamma), th_min ** (1 - 2 * gamma)) * (1 + abs(rho))
    return (0 < gamma < 1) and (c1 < 4) and (c2 <= 4)


def fit_ssvi(kq, tq, wq, wtq):
    taus = np.unique(tq)
    t_lo_, t_hi_ = float(tq.min()), float(tq.max())
    th_t, th_v = [], []
    for t in taus:
        kk_, ww_ = kq[tq == t], wq[tq == t]
        j = int(np.argmin(np.abs(kk_)))
        if abs(float(kk_[j])) <= 0.20 and float(ww_[j]) > 0:
            th_t.append(float(t)); th_v.append(float(ww_[j]))
    if len(th_t) >= 2:
        A = np.vstack([np.ones(len(th_t)), np.log(th_t)]).T
        coef, *_ = np.linalg.lstsq(A, np.log(th_v), rcond=None)
        a0, b0 = float(np.exp(coef[0])), float(np.clip(coef[1], 0.3, 1.5))
    else:
        a0, b0 = float(np.median(wq / tq ** 0.95)), 0.95

    def sse(p):
        rho, eta, gamma, alpha, beta = p
        rho = float(np.clip(rho, -0.95, 0.95)); eta = max(eta, 1e-4)
        gamma = float(np.clip(gamma, 0.05, 0.95)); alpha = max(alpha, 1e-6)
        beta = float(np.clip(beta, 0.3, 1.5))
        tot = sum(float(np.sum(wtq[tq == t] * (ssvi_w(kq[tq == t], alpha * t ** beta,
                                                      rho, eta, gamma) - wq[tq == t]) ** 2))
                  for t in taus)
        if not gj_ok(rho, eta, gamma, alpha, beta, t_lo_, t_hi_):
            tot += 1e3
        return tot

    best = None
    for x0 in [(-0.5, 1.0, 0.3, a0, b0), (-0.7, 0.6, 0.4, a0, b0)]:
        r = minimize(sse, x0, method="Nelder-Mead", options={"maxiter": 1200})
        if best is None or r.fun < best.fun:
            best = r
    rho, eta, gamma, alpha, beta = best.x
    p = dict(rho=float(np.clip(rho, -0.95, 0.95)), eta=float(max(eta, 1e-4)),
             gamma=float(np.clip(gamma, 0.05, 0.95)), alpha=float(max(alpha, 1e-6)),
             beta=float(np.clip(beta, 0.3, 1.5)))
    assert gj_ok(**p, t_lo=t_lo_, t_hi=t_hi_), "SSVI escaped the GJ region"
    return p


def ssvi_pack(pp, k_lo, k_hi, t_lo, t_hi, date):
    """SSVI surface pack with (near-)exact derivatives: tiny-step central differences on the
    ANALYTIC function -- accurate to ~1e-9, i.e. exact for every purpose here."""
    kg = np.linspace(k_lo, k_hi, 81); tg = np.linspace(t_lo, t_hi, 41)
    KK, TT = np.meshgrid(kg, tg)
    f = lambda K, T: ssvi_w(K, pp["alpha"] * T ** pp["beta"], pp["rho"], pp["eta"], pp["gamma"])
    h = 1e-5
    W = f(KK, TT)
    Wk = (f(KK + h, TT) - f(KK - h, TT)) / (2 * h)
    Wkk = (f(KK + h, TT) - 2 * W + f(KK - h, TT)) / h ** 2
    Wt = (f(KK, TT + h) - f(KK, TT - h)) / (2 * h)
    G = durrleman_g(KK, np.maximum(W, 1e-12), Wk, Wkk)
    return SurfacePack(kg, tg, W, Wt, G, model="ssvi", date=date)


# ---------- quotes loader (defensive on schema and date encodings) ----------
def load_day_quotes(date):
    if not REAL_PARQUET.exists():
        return None
    date_key = canonical_date(date)
    compact = date_key.replace("-", "")
    scan = pl.scan_parquet(REAL_PARQUET)
    cols = scan.collect_schema().names()
    iv_col = "iv_om" if "iv_om" in cols else "impl_volatility"
    if iv_col not in cols:
        raise KeyError("Neither `iv_om` nor `impl_volatility` exists in the clean parquet.")
    want = [c for c in ["date", "exdate", "tau", "k", iv_col, "cp_flag",
                        "strike_price", "strike", "best_bid", "best_offer",
                        "mid", "spread", "forward_price"] if c in cols]
    dtext = pl.col("date").cast(pl.Utf8)
    date_match = ((dtext.str.slice(0, 10) == date_key)
                  | (dtext.str.replace_all("-", "").str.slice(0, 8) == compact))
    otm_match = pl.col("is_otm") if "is_otm" in cols else pl.lit(True)
    df = (scan.filter(date_match & otm_match)
              .select(want)
              .drop_nulls([iv_col, "k", "tau"])
              .collect(engine="streaming"))
    return df.rename({iv_col: "iv"}) if df.height else None


# validate the FD field construction against analytic SSVI (bounds the NB04-pack asymmetry)
_pp = dict(rho=-0.55, eta=0.9, gamma=0.42, alpha=0.045, beta=0.95)
_kv = np.linspace(-0.5, 0.35, 321)
_tv = tau_hybrid(0.06, 1.5, 61)
_KV, _TV = np.meshgrid(_kv, _tv)
_fw = lambda K, T: ssvi_w(K, _pp["alpha"] * T ** _pp["beta"], _pp["rho"], _pp["eta"], _pp["gamma"])
_h = 1e-5
_Wv = _fw(_KV, _TV)
_Wtv = (_fw(_KV, _TV + _h) - _fw(_KV, _TV - _h)) / (2 * _h)
_Gv = durrleman_g(_KV, np.maximum(_Wv, 1e-12),
                  (_fw(_KV + _h, _TV) - _fw(_KV - _h, _TV)) / (2 * _h),
                  (_fw(_KV + _h, _TV) - 2 * _Wv + _fw(_KV - _h, _TV)) / _h ** 2)
_ref = SurfacePack(_kv, _tv, _Wv, _Wtv, _Gv, model="ref")
_fd = SurfacePack(_ref.k, _ref.t, _ref.W, model="fd_check")   # forces FD reconstruction
_int = (slice(2, -2), slice(2, -2))
err_g = np.max(np.abs(_fd.G[_int] - _ref.G[_int]))
err_wt = np.max(np.abs(_fd.Wt[_int] - _ref.Wt[_int]))
print(f"FD-vs-analytic field validation (interior): |g err| = {err_g:.2e}, "
      f"|wt err| = {err_wt:.2e}  (must be << {VIOL_TOL_MAT})")
assert err_g < VIOL_TOL_MAT / 5 and err_wt < VIOL_TOL_MAT / 5, \
    "FD grid too coarse for material-level auditing; increase export resolution"

In [ ]:
# ## 0c. VIX csv preparation (gated: runs only if the CBOE raw file is present)
if VIX_RAW.exists():
    raw = pl.read_csv(VIX_RAW, infer_schema_length=0)      # all-Utf8: no mixed-type inference
    _cmap = {c.strip().upper(): c for c in raw.columns}
    vix_df = (
        raw.select(
            pl.col(_cmap["DATE"]).str.strip_chars()
              .str.strptime(pl.Date, "%m/%d/%Y", strict=False).alias("d"),
            pl.col(_cmap["CLOSE"]).str.strip_chars().cast(pl.Float64, strict=False).alias("vix"),
        )
        .drop_nulls()
        .filter(pl.col("vix") > 0)
        .unique(subset="d", keep="last")                   # CBOE files occasionally repeat a day
        .sort("d")
        .filter(pl.col("d") >= pl.date(2018, 1, 1))
        .select(pl.col("d").cast(pl.Utf8).alias("date"), "vix")
    )
    _lo, _hi = float(vix_df["vix"].min()), float(vix_df["vix"].max())
    assert 5 < _lo and _hi < 200, \
        f"VIX not in index points (min {_lo}, max {_hi}) — section D divides by 100 itself"
    VIX_CSV.parent.mkdir(parents=True, exist_ok=True)
    vix_df.write_csv(VIX_CSV)
    print(f"[0c] wrote {VIX_CSV}: {vix_df.height} rows | {vix_df['date'][0]} -> "
          f"{vix_df['date'][-1]} | vix in [{_lo:.2f}, {_hi:.2f}]")
else:
    print(f"[0c] {VIX_RAW} absent — using existing {VIX_CSV}: {VIX_CSV.exists()}")

## A. Dupire local volatility — the constraints priced

$$ \sigma^2_{loc}(k,\tau) = \frac{\partial_\tau w(k,\tau)}{g(k,\tau)} $$
**Numerator = calendar constraint, denominator = butterfly constraint**: a violated surface
produces a negative or unbounded local variance that must be *repaired* before one MC path
can be drawn. The repair rate is the headline metric. Protocol follows Chataigner–Crépey–
Dixon §7: validity, day-to-day stability, MC repricing error. Two repricing errors:
**round-trip** (MC vs the surface's own prices: dynamic self-consistency) and **market**
(MC vs held-out quote IVs).

In [ ]:
def local_variance(pack):
    v = pack.Wt / np.where(np.abs(pack.G) < 1e-12, np.nan, pack.G)
    invalid = ~np.isfinite(v) | (v <= 0) | (pack.G <= 0) | (pack.Wt < 0)
    v_rep = np.clip(np.nan_to_num(v, nan=V_LOC_FLOOR), V_LOC_FLOOR, V_LOC_CAP)
    return v, v_rep, invalid


def mc_reprice(pack, k_eval, t_eval, n_paths=MC_PATHS, n_steps=MC_STEPS, seed=SEED):
    """Forward-measure local-vol MC: dX = -1/2 v_loc dt + sqrt(v_loc) dW, X0 = 0.
    Time grid = UNION of a uniform n_steps grid and the exact evaluation times, so snapshots
    land ON t_eval (the nearest-snapshot lookup accrued up to v*dt ~ 6e-4 of spurious total
    variance -- the same order as VIOL_TOL_MAT). Below the pack's tau_min, v_loc held flat."""
    _, v_rep, _ = local_variance(pack)
    interp = RegularGridInterpolator((pack.t, pack.k), v_rep, bounds_error=False, fill_value=None)
    t_hit = np.sort(np.unique(np.round(np.asarray(t_eval, float), 10)))
    T = float(t_hit[-1])
    edges = np.union1d(np.linspace(0.0, T, n_steps + 1), t_hit)
    hitset = set(t_hit.tolist())
    r = np.random.default_rng(seed)
    X = np.zeros(n_paths)
    snaps = {}
    for t0_, t1_ in zip(edges[:-1], edges[1:]):
        dt = float(t1_ - t0_)
        if dt <= 0:
            continue
        tq_ = np.clip(t0_, pack.t[0], pack.t[-1])
        v = interp(np.stack([np.full(n_paths, tq_), np.clip(X, pack.k[0], pack.k[-1])], -1))
        v = np.clip(v, V_LOC_FLOOR, V_LOC_CAP)
        X = X - 0.5 * v * dt + np.sqrt(v * dt) * r.standard_normal(n_paths)
        if float(t1_) in hitset:
            snaps[float(t1_)] = X.copy()
    w_mc = np.empty(len(k_eval))
    for i, (k, t) in enumerate(zip(k_eval, t_eval)):
        Xt = snaps[float(np.round(t, 10))]
        if k <= 0:
            price = np.mean(np.maximum(np.exp(k) - np.exp(Xt), 0.0))
            w_mc[i] = implied_w(np.array([max(price, 1e-10)]), np.array([k]), np.array([False]))[0]
        else:
            price = np.mean(np.maximum(np.exp(Xt) - np.exp(k), 0.0))
            w_mc[i] = implied_w(np.array([max(price, 1e-10)]), np.array([k]), np.array([True]))[0]
    return w_mc


rows_A = []
for date in DATES:
    q = load_day_quotes(date)
    day_packs = {m: p for (m, d), p in packs.items() if d == date and not m.endswith("lam0")}
    if q is not None:
        kq, tq, ivq = q["k"].to_numpy(), q["tau"].to_numpy(), q["iv"].to_numpy()
        wq = ivq ** 2 * tq
        wtq = 1.0 / (4 * np.maximum(wq, 1e-10) * tq); wtq = wtq / wtq.mean()
        day_packs["ssvi"] = ssvi_pack(fit_ssvi(kq, tq, wq, wtq),
                                      kq.min(), kq.max(), tq.min(), tq.max(), date)
    for m, pack in sorted(day_packs.items()):
        v, v_rep, invalid = local_variance(pack)
        repair_pct = float(100 * invalid.mean())
        if q is not None:
            hold = holdout_mask(date, len(kq))
            base = hold if hold.any() else np.ones(len(kq), bool)
            # no repricing beyond the pack's certified tau range (extrapolated local vol
            # is not the surface); k clipped only within the already in-range subset.
            in_dom = base & (tq >= pack.t[0]) & (tq <= pack.t[-1])
            sel = np.where(in_dom)[0]
            sel = sel[np.argsort(np.abs(kq[sel]))][:120]      # densest-information subset
            if len(sel) < 10:
                rt = mk = None
            else:
                ke, te, ive = kq[sel], tq[sel], ivq[sel]
                ke = np.clip(ke, pack.k[0] + 1e-6, pack.k[-1] - 1e-6)
                w_mc = mc_reprice(pack, ke, te)
                iv_mc = np.sqrt(w_mc / te)
                iv_surf = pack.iv(ke, te)
                rt = float(np.sqrt(np.mean((iv_mc - iv_surf) ** 2)) * 100)
                mk = float(np.sqrt(np.mean((iv_mc - ive) ** 2)) * 100)
        else:
            rt = mk = None
        rows_A.append(dict(model=m, date=date, repair_pct=repair_pct,
                           min_vloc=float(np.nanmin(v)), max_vloc=float(np.nanmax(v)),
                           mc_roundtrip_rmse_vp=rt, mc_market_rmse_vp=mk,
                           exact_fields=pack.exact))
        print(f"{date} [{m:>26}] repair {repair_pct:5.1f}% of grid | "
              f"v_loc in [{np.nanmin(v):+.4f},{np.nanmax(v):+.4f}] | "
              f"MC round-trip {rt if rt is None else f'{rt:.3f}'} vp | "
              f"MC vs market {mk if mk is None else f'{mk:.3f}'} vp | "
              f"{'exact' if pack.exact else 'FD'} fields")

if rows_A:
    dfA = pl.DataFrame(rows_A)
    dfA.write_parquet(OUT_DIR / "nb05_local_vol.parquet")
    print("written:", OUT_DIR / "nb05_local_vol.parquet")

# day-over-day local-vol stability (needs >= 2 consecutive dates per model)
stab = []
for m in {mm for (mm, _) in packs}:
    ds = sorted(d for (mm, d) in packs if mm == m)
    for d0, d1 in zip(ds[:-1], ds[1:]):
        p0, p1 = packs[(m, d0)], packs[(m, d1)]
        kg = np.linspace(max(p0.k[0], p1.k[0]), min(p0.k[-1], p1.k[-1]), 41)
        tg = np.linspace(max(p0.t[0], p1.t[0]), min(p0.t[-1], p1.t[-1]), 21)
        KK, TT = np.meshgrid(kg, tg)
        s0 = np.sqrt(np.clip(local_variance(p0)[1], V_LOC_FLOOR, V_LOC_CAP))
        s1 = np.sqrt(np.clip(local_variance(p1)[1], V_LOC_FLOOR, V_LOC_CAP))
        i0 = RegularGridInterpolator((p0.t, p0.k), s0)(np.stack([TT, KK], -1))
        i1 = RegularGridInterpolator((p1.t, p1.k), s1)(np.stack([TT, KK], -1))
        stab.append(dict(model=m, d0=d0, d1=d1,
                         mean_abs_shift=float(np.mean(np.abs(i1 - i0)) * 100)))
if stab:
    print(pl.DataFrame(stab))

# local-vol heatmaps for the latest date
if DATES:
    d = DATES[-1]
    show = {m: p for (m, dd), p in packs.items() if dd == d and not m.endswith("lam0")}
    if show:
        fig = make_subplots(rows=1, cols=len(show),
                            subplot_titles=[f"{m}: sigma_loc (red = repaired)" for m in sorted(show)])
        for j, m in enumerate(sorted(show), start=1):
            p = show[m]
            v, v_rep, invalid = local_variance(p)
            fig.add_trace(go.Heatmap(z=np.sqrt(v_rep), x=p.k, y=p.t, colorscale="Viridis",
                                     showscale=(j == len(show)), colorbar=dict(title="sigma_loc")), 1, j)
            M = np.where(invalid, 1.0, np.nan)
            fig.add_trace(go.Heatmap(z=M, x=p.k, y=p.t, showscale=False,
                                     colorscale=[[0, "rgba(214,39,40,0.85)"], [1, "rgba(214,39,40,0.85)"]]), 1, j)
        fig.update_layout(width=380 * len(show) + 120, height=420,
                          title=f"Dupire local volatility by family — {d} "
                                f"(red cells: v_loc invalid, repaired before simulation)")
        fig.show()

## C. Risk-neutral densities (Breeden–Litzenberger) — butterfly in probability units

$q(k) = g(k)\,\varphi(d_2)/\sqrt{w}$ with $d_2 = -k/\sqrt{w} - \sqrt{w}/2$: **positivity ⇔
butterfly**, so negative mass is the violation in the units a risk manager reads. Also
checked: $\int q\,dk \approx 1$ and $\int e^k q\,dk \approx 1$ (deviations on the truncated
export domain are reported as truncation, not error).

In [ ]:
def bl_density(pack, tau):
    kg = pack.k
    w = pack.w(kg, np.full_like(kg, tau))
    g = pack.g(kg, np.full_like(kg, tau))
    sw = np.sqrt(w)
    d2 = -kg / sw - sw / 2
    q = g * _phi(d2) / sw
    return kg, q


rows_C = []
for date in DATES:
    day_packs = {m: p for (m, d), p in packs.items() if d == date and not m.endswith("lam0")}
    for m, pack in sorted(day_packs.items()):
        for tau in np.quantile(pack.t, [0.15, 0.5, 0.9]):
            kg, q = bl_density(pack, float(tau))
            dk = kg[1] - kg[0]
            rows_C.append(dict(model=m, date=date, tau=float(tau),
                               neg_mass_pct=float(100 * np.sum(np.abs(q[q < 0])) * dk),
                               total_mass=float(np.sum(q) * dk),
                               martingale=float(np.sum(np.exp(kg) * q) * dk),
                               min_q=float(q.min())))
if rows_C:
    dfC = pl.DataFrame(rows_C)
    dfC.write_parquet(OUT_DIR / "nb05_densities.parquet")
    print(dfC)

if DATES:
    d = DATES[-1]
    fig = go.Figure()
    for m, pack in sorted({m: p for (m, dd), p in packs.items()
                           if dd == d and not m.endswith("lam0")}.items()):
        tau = float(np.quantile(pack.t, 0.5))
        kg, q = bl_density(pack, tau)
        fig.add_trace(go.Scatter(x=kg, y=q, name=f"{m} (tau={tau:.2f})"))
    fig.add_hline(y=0, line_dash="dot")
    fig.update_layout(width=880, height=420, xaxis_title="log-moneyness k",
                      yaxis_title="risk-neutral density q(k)",
                      title=f"Breeden–Litzenberger densities — {d} "
                            f"(any excursion below 0 is butterfly arbitrage)")
    fig.show()

## E. Economic significance of smoothing residuals — tightly framed

**Claim under test** (and the only one): residuals $r_i = \text{IV}^{mkt}_i -
\text{IV}^{model}_i$ on *held-out, in-domain* quotes contain economically meaningful
information, and that information degrades when the surface is arbitrage-dirty (λ=0
counterfactual). **Not an alpha claim.** Protocol per (day, family): residuals on NB05's
held-out quotes strictly inside the pack's (k,τ) box (out-of-domain quotes have no residual
— they are dropped, never clipped); match to day t+1 by `(exdate, strike)`;
IC = Spearman(r, −ΔIV); decile spread net of entry half-spreads, cost-capped universe.

In [ ]:
TRADE_COST_CAP_VP = 5.0
MIN_SIGNAL_NAMES = int(os.environ.get("NB05_MIN_SIGNAL_NAMES", "30"))


def parquet_date_pairs():
    """Sorted [(canonical_date, raw parquet date)] with duplicate canonical dates removed."""
    if not REAL_PARQUET.exists():
        return []
    raw = (pl.scan_parquet(REAL_PARQUET)
             .select("date").unique()
             .collect(engine="streaming")["date"].to_list())
    by_key = {}
    for x in raw:
        by_key.setdefault(canonical_date(x), x)
    return sorted(by_key.items())


def day_pair(date):
    """This date's quotes and the next trading day's quotes; (q0, q1, failure_reason)."""
    pairs = parquet_date_pairs()
    keys = [k for k, _ in pairs]
    date_key = canonical_date(date)
    if date_key not in keys:
        return None, None, "pack date absent from parquet"
    pos = keys.index(date_key)
    if pos + 1 >= len(keys):
        return None, None, "no next trading day in parquet"
    q0 = load_day_quotes(keys[pos])
    q1 = load_day_quotes(keys[pos + 1])
    if q0 is None:
        return None, None, "current day has no usable OTM quotes"
    if q1 is None:
        return None, None, "next day has no usable OTM quotes"
    return q0, q1, None


def in_domain(pack, kq, tq):
    """Quotes inside the pack's exported (k, tau) box; no clipping. Outside the box the
    surface is extrapolation: clipping k to the boundary and differencing against a far-wing
    quote manufactures residuals of tens of vol points that dominate every decile."""
    return ((kq >= pack.k[0]) & (kq <= pack.k[-1])
            & (tq >= pack.t[0]) & (tq <= pack.t[-1]))


def iv_half_spread(q, kq, wq, tq):
    """Per-quote IV half-spread from the real quote spread (parquet `spread`, else
    best_offer - best_bid) converted through Black vega with the day's forward."""
    if "forward_price" in q.columns and ({"spread"} <= set(q.columns)
                                         or {"best_bid", "best_offer"} <= set(q.columns)):
        F = q["forward_price"].to_numpy()
        spread = (q["spread"].to_numpy() if "spread" in q.columns
                  else q["best_offer"].to_numpy() - q["best_bid"].to_numpy())
        spread = np.maximum(np.asarray(spread, float), 0.0)
        vega = np.maximum(F * _phi((-kq + wq / 2) / np.sqrt(np.maximum(wq, 1e-12)))
                          * np.sqrt(tq), 1e-8)
        return spread / (2 * vega), "quote half-spreads"
    return np.full(len(kq), FLAT_COST_VP / 100), f"FLAT {FLAT_COST_VP} vp fallback"


def signal_universe(pack, nxt, ex0, st0, kq, tq, ivq, cost_iv, hold):
    """Shared E universe: held-out, in-domain, next-day-matched, cost-capped.
    Returns ((residual, next-day dIV, cost), diagnostics)."""
    dom = in_domain(pack, kq, tq)
    idx = np.where(hold & dom)[0]
    info = dict(n_quotes=int(len(kq)), n_hold=int(hold.sum()), n_domain=int(dom.sum()),
                n_hold_domain=int(len(idx)), n_matched=0, n_tradable=0, reason="")
    if len(idx) < MIN_SIGNAL_NAMES:
        info["reason"] = f"held-out in-domain < {MIN_SIGNAL_NAMES}"
        return None, info
    r = ivq[idx] - pack.iv(kq[idx], tq[idx])               # + = quote rich vs surface
    div, keep = [], []
    for j, i in enumerate(idx):
        key = option_key(ex0[i], st0[i])
        if key in nxt:
            div.append(nxt[key] - ivq[i]); keep.append(j)
    info["n_matched"] = int(len(keep))
    if len(keep) < MIN_SIGNAL_NAMES:
        info["reason"] = f"next-day matched < {MIN_SIGNAL_NAMES}"
        return None, info
    r_ = r[keep]
    div_ = np.asarray(div, float)
    c_ = np.asarray(cost_iv[idx][keep], float)
    tradable = np.isfinite(c_) & (c_ <= TRADE_COST_CAP_VP / 100)
    info["n_tradable"] = int(tradable.sum())
    if tradable.sum() < MIN_SIGNAL_NAMES:
        info["reason"] = f"tradable after cost cap < {MIN_SIGNAL_NAMES}"
        return None, info
    info["reason"] = "OK"
    return (r_[tradable], div_[tradable], c_[tradable]), info


_pack_dates = sorted(set(DATES))
_parquet_dates = [k for k, _ in parquet_date_pairs()]
_overlap = sorted(set(_pack_dates) & set(_parquet_dates))
print(f"[E preflight] pack dates={len(_pack_dates)} | parquet dates={len(_parquet_dates)} "
      f"| overlap={len(_overlap)}")

rows_E, diag_E = [], []
for date in DATES:
    q0, q1, pair_reason = day_pair(date)
    if pair_reason is not None:
        diag_E.append(dict(model="--", date=date, reason=pair_reason, n_quotes=0, n_hold=0,
                           n_domain=0, n_hold_domain=0, n_matched=0, n_tradable=0))
        continue
    strike_col = next((c for c in ["strike", "strike_price"] if c in q0.columns), None)
    if strike_col is None or "exdate" not in q0.columns:
        diag_E.append(dict(model="--", date=date, reason="missing exdate/strike columns",
                           n_quotes=q0.height, n_hold=0, n_domain=0, n_hold_domain=0,
                           n_matched=0, n_tradable=0))
        continue
    kq, tq, ivq = q0["k"].to_numpy(), q0["tau"].to_numpy(), q0["iv"].to_numpy()
    hold = holdout_mask(date, len(kq))
    if not hold.any():
        diag_E.append(dict(model="--", date=date, reason="empty holdout", n_quotes=len(kq),
                           n_hold=0, n_domain=0, n_hold_domain=0, n_matched=0, n_tradable=0))
        continue
    nxt = {option_key(e, s): float(v) for e, s, v in
           zip(q1["exdate"].to_list(), q1[strike_col].to_list(), q1["iv"].to_list())}
    ex0, st0 = q0["exdate"].to_list(), q0[strike_col].to_list()

    day_models = {m: p for (m, d), p in packs.items() if d == canonical_date(date)}
    wq = ivq ** 2 * tq
    wtq = 1.0 / (4 * np.maximum(wq, 1e-10) * tq); wtq = wtq / wtq.mean()
    if "ssvi" not in day_models:
        day_models["ssvi"] = ssvi_pack(fit_ssvi(kq, tq, wq, wtq),
                                       kq.min(), kq.max(), tq.min(), tq.max(),
                                       canonical_date(date))
    cost_iv, cost_note = iv_half_spread(q0, kq, wq, tq)

    for m, pack in sorted(day_models.items()):
        uni, info = signal_universe(pack, nxt, ex0, st0, kq, tq, ivq, cost_iv, hold)
        diag_E.append(dict(model=m, date=canonical_date(date), **info))
        if uni is None:
            continue
        r_, div_, c_ = uni
        ic = float(spearmanr(r_, -div_).statistic)
        qlo, qhi = np.quantile(r_, [0.1, 0.9])
        long_, short_ = r_ <= qlo, r_ >= qhi                # long cheap, short rich
        gross = float(np.mean(div_[long_]) - np.mean(div_[short_])) * 100
        costs = float(np.mean(c_[long_]) + np.mean(c_[short_])) * 100
        rows_E.append(dict(model=m, date=canonical_date(date), n_matched=len(r_), ic=ic,
                           decile_spread_gross_vp=gross, entry_costs_vp=costs,
                           decile_spread_net_vp=gross - costs, cost_basis=cost_note))
        print(f"{canonical_date(date)} [{m:>26}] n={len(r_):>4} | IC {ic:+.3f} | decile spread "
              f"{gross:+.3f} vp gross / {gross - costs:+.3f} net ({cost_note})")

if rows_E:
    dfE = pl.DataFrame(rows_E)
    dfE.write_parquet(OUT_DIR / "nb05_residual_economics.parquet")
    if any(m.endswith("lam0") for m, _ in packs):
        cmp_ = dfE.filter(pl.col("model").str.contains("deep")).sort(["date", "model"])
        print("\nCLEAN (lam=10) vs DIRTY (lam=0) signal quality — same day, same quotes:")
        print(cmp_.select(["date", "model", "ic", "decile_spread_net_vp"]))
    n_days = dfE["date"].n_unique()
    if n_days < 20:
        print(f"\n[power] Only {n_days} day(s): pipeline validation, not statistical evidence.")
else:
    print("[E] no model-day produced a usable universe.")

if diag_E:
    diagE = pl.DataFrame(diag_E)
    badE = diagE.filter(pl.col("reason") != "OK")
    if badE.height:
        print("\n[E diagnostic] drop reasons:")
        print(badE.group_by("reason").len().sort("len", descending=True))

## F. Trading the violations — executable arbitrage and residual relative value

Two questions, kept strictly apart.

**F1 — Is there executable static arbitrage in the QUOTES?** Model-free checks inside each
expiry, priced at the touch (buy at the offer, sell at the bid): vertical monotonicity and
butterfly convexity, calls and puts separately. The same checks at MID prices run alongside:
the gap between the two columns is the bid-ask spread absorbing the "violations".

**F2 — Is the smoothing residual a tradable SIGNAL?** Not arbitrage: statistical relative
value on a **common universe** (identical names and costs across the day's models — only
the surface differs), bucket-neutral ranking (τ-terciles × |k|-terciles), an **entry
filter** (trade only quotes mispriced beyond their own half-spread — the F1 bridge), swept
over holding horizons × an execution ladder (touch / half / mid). `mid` bounds the
market-maker capture of the signal; `touch` is what a taker pays. First-order vega P&L,
one round trip per holding period, non-overlapping periods. A research backtest, not a
trading system.

In [ ]:
F1_MAX_DAYS = int(os.environ.get("NB05_F1_MAX_DAYS", "250"))
F_GATE_VP = 5.0            # same fit-validity gate as NB04


# ---------------- F1: executable static-arbitrage scanner (bid/ask, within expiry) -----
def _side_checks(K, bid, ask, is_call):
    """Vertical + butterfly checks on one (expiry, type) side, strikes ascending.
    Executable convention: BUY at ask, SELL at bid; a violation is a portfolio with
    payoff >= 0 in every state and strictly positive premium received at entry."""
    out = dict(n=len(K), n_pairs=max(len(K) - 1, 0), n_tris=max(len(K) - 2, 0),
               vert_exec=0, vert_mid=0, bfly_exec=0, bfly_mid=0, exec_premium=0.0)
    if len(K) < 2:
        return out
    mid = 0.5 * (bid + ask)
    if is_call:   # C must be decreasing in K
        v_exec = bid[1:] - ask[:-1]
        v_mid = mid[1:] - mid[:-1]
    else:         # P must be increasing in K
        v_exec = bid[:-1] - ask[1:]
        v_mid = mid[:-1] - mid[1:]
    out["vert_exec"] = int(np.sum(v_exec > 0))
    out["vert_mid"] = int(np.sum(v_mid > 0))
    out["exec_premium"] += float(np.sum(v_exec[v_exec > 0]))
    if len(K) >= 3:
        K1, K2, K3 = K[:-2], K[1:-1], K[2:]
        l1 = (K3 - K2) / (K3 - K1)
        l3 = (K2 - K1) / (K3 - K1)
        c_exec = l1 * ask[:-2] - bid[1:-1] + l3 * ask[2:]
        c_mid = l1 * mid[:-2] - mid[1:-1] + l3 * mid[2:]
        out["bfly_exec"] = int(np.sum(c_exec < 0))
        out["bfly_mid"] = int(np.sum(c_mid < 0))
        out["exec_premium"] += float(np.sum(-c_exec[c_exec < 0]))
    return out


def scan_static_arb(dates):
    cols = pl.scan_parquet(REAL_PARQUET).collect_schema().names()
    need = {"date", "exdate", "cp_flag", "strike", "best_bid", "best_offer"}
    if not need <= set(cols):
        print(f"[F1] missing columns {sorted(need - set(cols))} — scanner skipped.")
        return None
    df = (pl.scan_parquet(REAL_PARQUET)
            .filter(pl.col("is_otm")
                    & pl.col("date").cast(pl.Utf8).is_in([str(d) for d in dates]))
            .select(["date", "exdate", "cp_flag", "strike", "best_bid", "best_offer"])
            .collect(engine="streaming"))
    rows = []
    for (d, ex, cp), sl in df.group_by(["date", "exdate", "cp_flag"], maintain_order=True):
        sl = sl.sort("strike")
        res = _side_checks(sl["strike"].to_numpy(), sl["best_bid"].to_numpy(),
                           sl["best_offer"].to_numpy(), str(cp) == "C")
        rows.append(dict(date=str(d), exdate=str(ex), cp=str(cp), **res))
    return pl.DataFrame(rows)


if REAL_PARQUET.exists():
    _all = (pl.scan_parquet(REAL_PARQUET).select("date").unique()
              .collect(engine="streaming")["date"].sort().to_list())
    pick = np.linspace(0, len(_all) - 1, min(F1_MAX_DAYS, len(_all))).round().astype(int)
    scan_dates = [_all[i] for i in sorted(set(pick.tolist()))]
    arb = scan_static_arb(scan_dates)
    if arb is not None and arb.height:
        n_checks = int(arb["n_pairs"].sum() + arb["n_tris"].sum())
        n_exec = int(arb["vert_exec"].sum() + arb["bfly_exec"].sum())
        n_mid = int(arb["vert_mid"].sum() + arb["bfly_mid"].sum())
        day_any = (arb.group_by("date")
                     .agg((pl.col("vert_exec") + pl.col("bfly_exec")).sum().alias("exec"),
                          (pl.col("vert_mid") + pl.col("bfly_mid")).sum().alias("mid")))
        arb.write_parquet(OUT_DIR / "nb05_static_arb.parquet")
        print(f"[F1] {len(scan_dates)} day(s), {n_checks:,} vertical/butterfly checks at the touch")
        print(f"[F1] EXECUTABLE violations : {n_exec:>8,}  "
              f"({100 * n_exec / max(n_checks, 1):.4f}% of checks; "
              f"{int((day_any['exec'] > 0).sum())}/{len(scan_dates)} day(s) with >= 1)")
        print(f"[F1] MID-price violations  : {n_mid:>8,}  "
              f"({100 * n_mid / max(n_checks, 1):.4f}% of checks; "
              f"{int((day_any['mid'] > 0).sum())}/{len(scan_dates)} day(s) with >= 1)")
        print(f"[F1] total premium locked by executable violations: "
              f"{arb['exec_premium'].sum():.4f} index points "
              f"(x100 multiplier = ${100 * arb['exec_premium'].sum():,.0f} per unit size, pre-fees)")
        print("[F1] The mid column minus the exec column is the spread doing its job.")
else:
    print("[F1] gated: clean parquet unavailable.")


# ---------------- F2: residual relative-value backtest --------------------------------
def surface_day_rmse(pack, kq, tq, ivq):
    """Pack RMSE on the day's in-domain quotes, in vol points."""
    m = in_domain(pack, kq, tq)
    if m.sum() < MIN_SIGNAL_NAMES:
        return float("inf")
    return float(np.sqrt(np.mean((pack.iv(kq[m], tq[m]) - ivq[m]) ** 2)) * 100)


HORIZONS = [int(x) for x in os.environ.get("NB05_F2_HORIZONS", "1,3,5,10").split(",")]
EXEC_LADDER = {"touch": 1.0, "half": 0.5, "mid": 0.0}   # fraction of the full spread paid
MIN_TRADE_NAMES = 20

_PDP_CACHE = None


def parquet_date_pairs_cached():
    global _PDP_CACHE
    if _PDP_CACHE is None:
        _PDP_CACHE = parquet_date_pairs()
    return _PDP_CACHE


def quotes_at_offset(date, h):
    """Quotes h trading days after `date`; (df, failure_reason)."""
    keys = [k for k, _ in parquet_date_pairs_cached()]
    dk = canonical_date(date)
    if dk not in keys:
        return None, "pack date absent from parquet"
    pos = keys.index(dk)
    if pos + h >= len(keys):
        return None, f"no t+{h} trading day in parquet"
    q = load_day_quotes(keys[pos + h])
    return (q, None) if q is not None else (None, f"t+{h} day has no usable quotes")


bt_rows, gated_days, drop_rows, precheck_F2 = [], [], [], []
for date in DATES:
    q0 = load_day_quotes(date)
    if q0 is None:
        precheck_F2.append(dict(date=canonical_date(date), reason="no usable OTM quotes"))
        continue
    strike_col = next((c for c in ["strike", "strike_price"] if c in q0.columns), None)
    if strike_col is None or "exdate" not in q0.columns:
        precheck_F2.append(dict(date=canonical_date(date), reason="missing exdate/strike columns"))
        continue
    kq, tq, ivq = q0["k"].to_numpy(), q0["tau"].to_numpy(), q0["iv"].to_numpy()
    wq = ivq ** 2 * tq
    hold = holdout_mask(date, len(kq))
    if not hold.any():
        precheck_F2.append(dict(date=canonical_date(date), reason="empty holdout"))
        continue
    ex0, st0 = q0["exdate"].to_list(), q0[strike_col].to_list()
    half_iv, cost_note = iv_half_spread(q0, kq, wq, tq)          # per-name HALF-spread, IV units

    day_models = {m: p for (m, d), p in packs.items() if d == canonical_date(date)}
    wtq = 1.0 / (4 * np.maximum(wq, 1e-10) * tq); wtq = wtq / wtq.mean()
    if "ssvi" not in day_models:
        day_models["ssvi"] = ssvi_pack(fit_ssvi(kq, tq, wq, wtq),
                                       kq.min(), kq.max(), tq.min(), tq.max(),
                                       canonical_date(date))

    # -------- COMMON universe: held-out quotes inside EVERY model's domain, finite cost.
    dom_all = np.ones(len(kq), bool)
    for p in day_models.values():
        dom_all &= in_domain(p, kq, tq)
    base = hold & dom_all & np.isfinite(half_iv) & (half_iv <= TRADE_COST_CAP_VP / 100)
    idx0 = np.where(base)[0]
    if len(idx0) < MIN_SIGNAL_NAMES:
        precheck_F2.append(dict(date=canonical_date(date), reason="common universe too small"))
        continue

    tb = np.digitize(tq[idx0], np.quantile(tq[idx0], [1 / 3, 2 / 3]))
    kb = np.digitize(np.abs(kq[idx0]), np.quantile(np.abs(kq[idx0]), [1 / 3, 2 / 3]))
    bucket0 = tb * 3 + kb

    for h in HORIZONS:
        qh, why = quotes_at_offset(date, h)
        if qh is None:
            precheck_F2.append(dict(date=canonical_date(date), reason=why))
            continue
        nxt = {option_key(e, s): float(v) for e, s, v in
               zip(qh["exdate"].to_list(), qh[strike_col].to_list(), qh["iv"].to_list())}
        jj, dv, bb = [], [], []
        for pos_i, i in enumerate(idx0):
            key = option_key(ex0[i], st0[i])
            if key in nxt:
                jj.append(i); dv.append(nxt[key] - ivq[i]); bb.append(bucket0[pos_i])
        if len(jj) < MIN_SIGNAL_NAMES:
            precheck_F2.append(dict(date=canonical_date(date),
                                    reason=f"t+{h} matched < {MIN_SIGNAL_NAMES}"))
            continue
        jj = np.asarray(jj); dv = np.asarray(dv, float); bb = np.asarray(bb)
        c_full = 2.0 * half_iv[jj]                    # full spread = one round trip per name

        for m, pack in sorted(day_models.items()):
            day_rmse = surface_day_rmse(pack, kq, tq, ivq)
            if day_rmse > F_GATE_VP:
                if h == HORIZONS[0]:
                    gated_days.append(dict(model=m, date=canonical_date(date),
                                           day_rmse_vp=day_rmse))
                continue
            r = ivq[jj] - pack.iv(kq[jj], tq[jj])     # + = quote rich vs surface
            rn = r.copy()                              # bucket-neutral ranking signal
            for b in np.unique(bb):
                rn[bb == b] -= rn[bb == b].mean()
            f = np.abs(r) > half_iv[jj]                # ENTRY FILTER: beyond own half-spread
            n_f = int(f.sum())
            if n_f < MIN_TRADE_NAMES:
                drop_rows.append(dict(model=m, date=canonical_date(date), h=h,
                                      reason="beyond-spread universe < min", n_filter=n_f))
                continue
            rr, dvv, cf = rn[f], dv[f], c_full[f]
            qlo, qhi = np.quantile(rr, [0.1, 0.9])
            L, S = rr <= qlo, rr >= qhi
            if L.sum() < 3 or S.sum() < 3:
                drop_rows.append(dict(model=m, date=canonical_date(date), h=h,
                                      reason="degenerate deciles", n_filter=n_f))
                continue
            gross = float(np.mean(dvv[L]) - np.mean(dvv[S])) * 100
            cost_touch = float(np.mean(cf[L]) + np.mean(cf[S])) * 100
            for ex_name, mult in EXEC_LADDER.items():
                bt_rows.append(dict(model=m, h=h, exec=ex_name, date=canonical_date(date),
                                    n_universe=len(jj), n_filter=n_f,
                                    n_traded=int(L.sum() + S.sum()),
                                    day_rmse_vp=day_rmse, pnl_gross_vp=gross,
                                    cost_vp=mult * cost_touch,
                                    pnl_net_vp=gross - mult * cost_touch,
                                    cost_basis=cost_note))

if precheck_F2:
    print("[F2 precheck] skips:")
    print(pl.DataFrame(precheck_F2).group_by("reason").len().sort("len", descending=True))
if gated_days:
    print(f"[F2 gate] {len(gated_days)} model-day(s) excluded "
          f"(in-domain surface RMSE > {F_GATE_VP} vp).")
if drop_rows:
    print("[F2 drop]:")
    print(pl.DataFrame(drop_rows).group_by(["reason"]).len().sort("len", descending=True))

if bt_rows:
    bt = pl.DataFrame(bt_rows).sort(["model", "h", "exec", "date"])
    bt.write_parquet(OUT_DIR / "nb05_backtest_daily.parquet")

    stats_rows = []
    for (m, h, ex), s in bt.group_by(["model", "h", "exec"], maintain_order=True):
        s = s.sort("date")
        take = list(range(0, s.height, int(h)))        # non-overlapping holding periods
        pnl = s["pnl_net_vp"].to_numpy()[take]
        g = s["pnl_gross_vp"].to_numpy()[take]
        n = len(pnl)
        mu = float(np.mean(pnl))
        sd = float(np.std(pnl, ddof=1)) if n > 1 else float("nan")
        ann = np.sqrt(252.0 / int(h))
        sharpe = float(mu / sd * ann) if sd == sd and sd > 0 else float("nan")
        cum = np.cumsum(pnl)
        maxdd = float(np.max(np.maximum.accumulate(cum) - cum)) if n else float("nan")
        stats_rows.append(dict(model=str(m), h=int(h), exec=str(ex), n_periods=n,
                               mean_n_filter=float(s["n_filter"].mean()),
                               mean_gross_vp=float(np.mean(g)),
                               mean_cost_vp=float(s["cost_vp"].to_numpy()[take].mean()),
                               mean_net_vp=mu, ann_sharpe=sharpe,
                               hit_rate=float(np.mean(pnl > 0)), max_drawdown_vp=maxdd))
    bt_stats = pl.DataFrame(stats_rows).sort(["exec", "model", "h"])
    bt_stats.write_parquet(OUT_DIR / "nb05_backtest_stats.parquet")

    for ex in ["touch", "half", "mid"]:
        piv = (bt_stats.filter(pl.col("exec") == ex)
                        .pivot(values="ann_sharpe", index="model", on="h").sort("model"))
        print(f"\n[F2] annualized Sharpe (net), execution = {ex} (cols = holding horizon):")
        print(piv)
        piv_n = (bt_stats.filter(pl.col("exec") == ex)
                          .pivot(values="mean_net_vp", index="model", on="h").sort("model"))
        print(f"[F2] mean net P&L per period (vp), execution = {ex}:")
        print(piv_n)
    print("\n[F2] full stats table (per model x horizon x execution):")
    print(bt_stats)

    b1 = bt.filter((pl.col("h") == 1) & (pl.col("exec") == "touch"))
    if b1.height:
        all_dates = b1["date"].unique().sort().to_list()
        pos = {d: i for i, d in enumerate(all_dates)}
        fig = go.Figure()
        for m in b1["model"].unique().sort().to_list():
            s = b1.filter(pl.col("model") == m).sort("date")
            fig.add_trace(go.Scatter(x=[pos[d] for d in s["date"].to_list()],
                                     y=np.cumsum(s["pnl_net_vp"].to_numpy()),
                                     mode="lines+markers", name=m,
                                     customdata=s["date"].to_list(),
                                     hovertemplate="%{customdata}<br>cum %{y:.2f} vp"
                                                   "<extra>" + m + "</extra>"))
        _d = [np.datetime64(d) for d in all_dates]
        gaps = [i for i in range(1, len(_d))
                if (_d[i] - _d[i - 1]).astype("timedelta64[D]").astype(int) > 5]
        for i in gaps:
            fig.add_vline(x=i - 0.5, line_dash="dash", line_color="rgba(120,120,120,0.7)")
        step = max(1, len(all_dates) // 12)
        fig.add_hline(y=0, line_dash="dot")
        fig.update_layout(width=920, height=440,
                          title="F2 — h=1 at the touch, common universe, bucket-neutral, "
                                "beyond-spread entries"
                                + (f" — {len(gaps)} calendar gap(s), dashed" if gaps else ""),
                          xaxis=dict(title="trading session", tickmode="array",
                                     tickvals=list(range(0, len(all_dates), step)),
                                     ticktext=[all_dates[i] for i in
                                               range(0, len(all_dates), step)],
                                     tickangle=-45),
                          yaxis_title="cumulative net P&L (vol pts)")
        fig.show()

    n_bt_days = bt["date"].n_unique()
    if n_bt_days < 60:
        avg_dec = bt.filter(pl.col("h") == 1)["n_traded"].mean()
        print(f"\n[power] {n_bt_days} day(s); average traded names/day ~ "
              f"{0 if avg_dec is None else avg_dec:.0f}. Pipeline validation, not evidence.")
    print("[F2] Reading grid: `mid` bounds the market-maker capture; `touch` is what a taker "
          "pays; horizons separate 1-day momentum from multi-day reversion. Clean-vs-dirty "
          "stays valid in every cell: identical universes, identical costs.")
else:
    print("[F2] no backtestable model-days — see the precheck/gate/drop tables above.")

## G. Detection power — the net-profitability frontier of this market

F2 measures the signal that *exists*; G measures the signal this pipeline and this market
**could detect and monetize if it existed**. Protocol: take a real day — real quotes, real
per-name half-spreads, and *measured* noise levels — and inject a synthetic mispricing of
amplitude $a$ (vol points, random sign) into a fraction of names, decaying toward the
surface with half-life $h$ (trading days). Run the injected residuals through the F2
selection machinery (entry filter, deciles) and compute net P&L at the touch over each
holding horizon. Sweeping $(a, h)$ traces the **frontier**: the minimum amplitude and
persistence a residual signal must have to clear real round-trip costs.

Two calibrations keep this honest: the observation noise $\sigma_{obs}$ is the robust
cross-sectional dispersion of *actual* residuals (a signal must be detected through it),
and the outcome noise $\sigma_\varepsilon$ is the robust dispersion of *actual* matched
day-over-day IV moves net of per-expiry common shifts. The comparison line: the share of
real quotes beyond their own half-spread, their median exceedance, and the empirical
residual half-life $\hat h$ from day-over-day residual autocorrelation. If the empirical
point sits below the frontier, "no net alpha" is a demonstrated property of the market's
spread structure — the quantitative twin of F1. Injecting a large, persistent signal also
validates the pipeline: it MUST light up, or F2 has a bug.

In [ ]:
G_INJ_FRAC = 0.10
G_A_GRID_VP = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0]
G_H_GRID = [1, 2, 3, 5, 8, 12]
G_HOLDS = sorted(set(HORIZONS))
G_NREP = int(os.environ.get("NB05_G_NREP", "120"))


def _robust_std(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 10:
        return float("nan")
    return float(1.4826 * np.median(np.abs(x - np.median(x))))


g_date = None
for date in reversed(DATES):
    _q0 = load_day_quotes(date)
    _q1, _ = quotes_at_offset(date, 1)
    if _q0 is not None and _q1 is not None:
        g_date = date
        break

if g_date is None:
    print("[G] gated: no date with a usable next-day chain.")
else:
    q0, q1 = load_day_quotes(g_date), quotes_at_offset(g_date, 1)[0]
    strike_col = next(c for c in ["strike", "strike_price"] if c in q0.columns)
    kq, tq, ivq = q0["k"].to_numpy(), q0["tau"].to_numpy(), q0["iv"].to_numpy()
    wq = ivq ** 2 * tq
    hold = holdout_mask(g_date, len(kq))
    half_iv, cost_note = iv_half_spread(q0, kq, wq, tq)
    wtq = 1.0 / (4 * np.maximum(wq, 1e-10) * tq); wtq = wtq / wtq.mean()
    ref = ssvi_pack(fit_ssvi(kq, tq, wq, wtq), kq.min(), kq.max(), tq.min(), tq.max(),
                    canonical_date(g_date))

    base = hold & np.isfinite(half_iv) & (half_iv <= TRADE_COST_CAP_VP / 100)
    U = np.where(base)[0]
    halfU = half_iv[U]
    costU = 2.0 * halfU                                  # full spread per round trip
    nU = len(U)

    # --- measured noise levels (the honesty of the exercise) ---
    r_real = ivq[U] - ref.iv(kq[U], tq[U])
    sig_obs = _robust_std(r_real)                        # cross-sectional residual dispersion
    nxt1 = {option_key(e, s): float(v) for e, s, v in
            zip(q1["exdate"].to_list(), q1[strike_col].to_list(), q1["iv"].to_list())}
    ex0, st0 = q0["exdate"].to_list(), q0[strike_col].to_list()
    dmoves, dkeys, dexp = [], [], []
    for i in U:
        key = option_key(ex0[i], st0[i])
        if key in nxt1:
            dmoves.append(nxt1[key] - ivq[i]); dkeys.append(i); dexp.append(key[0])
    dmoves = np.asarray(dmoves, float)
    # per-expiry common shift removed -> idiosyncratic daily IV noise
    idio = dmoves.copy()
    dexp = np.asarray(dexp)
    for e in np.unique(dexp):
        idio[dexp == e] -= np.median(idio[dexp == e])
    sig_eps = _robust_std(idio)

    # empirical persistence: residual today vs residual tomorrow (vs tomorrow's own surface)
    kq1, tq1, ivq1 = q1["k"].to_numpy(), q1["tau"].to_numpy(), q1["iv"].to_numpy()
    wq1 = ivq1 ** 2 * tq1
    wtq1 = 1.0 / (4 * np.maximum(wq1, 1e-10) * tq1); wtq1 = wtq1 / wtq1.mean()
    ref1 = ssvi_pack(fit_ssvi(kq1, tq1, wq1, wtq1), kq1.min(), kq1.max(),
                     tq1.min(), tq1.max(), "t+1")
    r1_map = {option_key(e, s): float(v) - float(ref1.iv(np.array([kk]), np.array([tt]))[0])
              for e, s, v, kk, tt in zip(q1["exdate"].to_list(), q1[strike_col].to_list(),
                                         q1["iv"].to_list(), kq1, tq1)}
    r0_l, r1_l = [], []
    for j, i in enumerate(U):
        key = option_key(ex0[i], st0[i])
        if key in r1_map:
            r0_l.append(r_real[j]); r1_l.append(r1_map[key])
    rho_emp = float(np.corrcoef(r0_l, r1_l)[0, 1]) if len(r0_l) > 50 else float("nan")
    h_emp = float(-1.0 / np.log2(rho_emp)) if rho_emp == rho_emp and 0 < rho_emp < 1 else 0.0

    beyond = np.abs(r_real) > halfU
    pct_beyond = 100 * float(beyond.mean())
    med_exceed_vp = (float(np.median((np.abs(r_real) - halfU)[beyond])) * 100
                     if beyond.any() else 0.0)

    print(f"[G] day {canonical_date(g_date)} | universe {nU} names ({cost_note}) | "
          f"sigma_obs {sig_obs*100:.2f} vp | sigma_eps(1d, idio) {sig_eps*100:.2f} vp")
    print(f"[G] empirical residuals: {pct_beyond:.1f}% beyond their own half-spread "
          f"(median exceedance {med_exceed_vp:.2f} vp) | day-1 autocorr {rho_emp:+.2f} "
          f"-> half-life ~ {h_emp:.1f} trading day(s)")

    rg = np.random.default_rng(SEED + 7)
    sharpe_grid = np.full((len(G_A_GRID_VP), len(G_H_GRID)), np.nan)
    bestj_grid = np.zeros_like(sharpe_grid)
    for ia, a_vp in enumerate(G_A_GRID_VP):
        a = a_vp / 100.0
        for ih, h in enumerate(G_H_GRID):
            best = -np.inf; bestj = 0
            for j in G_HOLDS:
                capture = a * (1.0 - 0.5 ** (j / h))
                pnls = np.empty(G_NREP)
                for rep in range(G_NREP):
                    s = rg.choice([-1.0, 1.0], nU) * (rg.random(nU) < G_INJ_FRAC)
                    r_obs = s * a + rg.normal(0.0, sig_obs, nU)
                    f = np.abs(r_obs) > halfU
                    if f.sum() < MIN_TRADE_NAMES:
                        pnls[rep] = np.nan
                        continue
                    rr, sf, cff = r_obs[f], s[f], costU[f]
                    qlo, qhi = np.quantile(rr, [0.1, 0.9])
                    L, S = rr <= qlo, rr >= qhi
                    div_true = -sf * capture             # mispricing decays toward surface
                    eps = rg.normal(0.0, sig_eps * np.sqrt(j), f.sum())
                    pnl = ((div_true[L] + eps[L]).mean() - (div_true[S] + eps[S]).mean()
                           - (cff[L].mean() + cff[S].mean()))
                    pnls[rep] = pnl * 100
                pnls = pnls[np.isfinite(pnls)]
                if len(pnls) < 20:
                    continue
                mu, sd = float(pnls.mean()), float(pnls.std(ddof=1))
                sh = mu / sd * np.sqrt(252.0 / j) if sd > 0 else np.nan
                if sh == sh and sh > best:
                    best, bestj = sh, j
            sharpe_grid[ia, ih] = best if best > -np.inf else np.nan
            bestj_grid[ia, ih] = bestj

    fig = go.Figure(go.Heatmap(z=sharpe_grid, x=[str(h) for h in G_H_GRID],
                               y=[f"{a}" for a in G_A_GRID_VP],
                               colorscale="RdBu", zmid=0.0,
                               colorbar=dict(title="ann Sharpe (net, touch)")))
    fig.add_trace(go.Contour(z=sharpe_grid, x=[str(h) for h in G_H_GRID],
                             y=[f"{a}" for a in G_A_GRID_VP],
                             contours=dict(start=0, end=0, coloring="none",
                                           showlabels=True),
                             line=dict(width=3, color="black"), showscale=False))
    fig.update_layout(width=760, height=480,
                      title=f"G — detection power frontier, {canonical_date(g_date)} "
                            f"(black contour: breakeven at the touch; empirical signal: "
                            f"{pct_beyond:.0f}% beyond spread, half-life ~{h_emp:.1f}d)",
                      xaxis_title="mispricing half-life h (trading days)",
                      yaxis_title="injected amplitude a (vol points)")
    fig.show()

    frontier = []
    for ih, h in enumerate(G_H_GRID):
        col = sharpe_grid[:, ih]
        ok = [G_A_GRID_VP[ia] for ia in range(len(G_A_GRID_VP))
              if col[ia] == col[ia] and col[ia] > 0]
        frontier.append(dict(half_life_days=h,
                             min_profitable_a_vp=(min(ok) if ok else float("nan")),
                             best_hold_days=int(bestj_grid[np.argmax(col == col), ih])))
    dfG = pl.DataFrame(frontier)
    print("[G] breakeven frontier (min amplitude net-profitable at the touch, per half-life):")
    print(dfG)
    dfG.write_parquet(OUT_DIR / "nb05_detection_frontier.parquet")

    chk = sharpe_grid[G_A_GRID_VP.index(5.0), G_H_GRID.index(12)]
    print(f"[G] pipeline validation: injected a=5 vp, h=12d -> ann Sharpe {chk:.1f} "
          f"({'PASS: the machinery detects large persistent signals; the empty real-data '
          'result is a property of the market, not of the code' if chk == chk and chk > 2
          else 'FAIL: F2 machinery cannot detect an obvious signal — debug before trusting '
          'any empty result'})")
    print(f"[G] verdict template for the thesis: a residual must exceed "
          f"~{np.nanmin(dfG['min_profitable_a_vp'].to_numpy()):.2f} vp with multi-day "
          f"persistence to clear touch costs; empirically {pct_beyond:.1f}% of SPX quotes "
          f"exceed their half-spread at all (median exceedance {med_exceed_vp:.2f} vp, "
          f"half-life ~{h_emp:.1f}d).")

## D. VIX replication (bonus, gated) — an external ground truth for the wings

$$ \sigma_{VS}^2(T)\,T = 2\int \tilde q(k)\,e^{-k}\,dk, \qquad
   \tilde q(k) = \text{normalized OTM Black price at } (k, w(k,T)), $$
with total variance interpolated to $T=30/365$, compared to the published VIX. The integral
runs over the **common k-strip of the day's models** so families at different export ranges
are comparable; the truncation deficit is itself informative — largest exactly when the
model's wings are poorest. Self-tested on flat Black-Scholes.

In [ ]:
def varswap_rate(pack, T=30 / 365, nk=400, k_lo=None, k_hi=None):
    t0 = float(np.clip(T, pack.t[0], pack.t[-1]))
    lo = pack.k[0] if k_lo is None else max(pack.k[0], k_lo)
    hi = pack.k[-1] if k_hi is None else min(pack.k[-1], k_hi)
    kg = np.linspace(lo + 1e-4, hi - 1e-4, nk)
    w = pack.w(kg, np.full_like(kg, t0)) * (T / t0)     # linear-in-tau total variance scaling
    otm = np.where(kg <= 0, black_put(kg, w), black_call(kg, w))
    _trapz = getattr(np, "trapezoid", getattr(np, "trapz", None))
    integ = 2 * _trapz(otm * np.exp(-kg), kg)
    return float(np.sqrt(max(integ, 1e-12) / T))


_flat = SurfacePack(np.linspace(-2.0, 2.0, 201), np.linspace(0.02, 1.0, 21),
                    0.04 * np.tile(np.linspace(0.02, 1.0, 21)[:, None], (1, 201)), model="flat")
_vs = varswap_rate(_flat)
print(f"VIX machinery self-test: flat sigma=0.200 -> varswap {_vs:.4f} (truncation-limited)")
assert abs(_vs - 0.20) < 5e-3

if VIX_CSV.exists() and DATES:
    vix = pl.read_csv(VIX_CSV).with_columns(pl.col("date").cast(pl.Utf8))
    rows_D = []
    for date in DATES:
        row = vix.filter(pl.col("date") == canonical_date(date))
        if not row.height:
            continue
        v_mkt = float(row["vix"][0]) / 100
        day_models = {m: p for (m, d), p in packs.items()
                      if d == date and not m.endswith("lam0")}
        if not day_models:
            continue
        k_lo = max(p.k[0] for p in day_models.values())
        k_hi = min(p.k[-1] for p in day_models.values())
        for m, pack in sorted(day_models.items()):
            mv = varswap_rate(pack, k_lo=k_lo, k_hi=k_hi)
            rows_D.append(dict(model=m, date=date, model_vs=mv, vix=v_mkt,
                               err_volpts=(mv - v_mkt) * 100,
                               k_strip=float(k_hi - k_lo)))
    if rows_D:
        dfD = pl.DataFrame(rows_D)
        dfD.write_parquet(OUT_DIR / "nb05_vix.parquet")
        print(dfD)
        summ = (dfD.group_by("model")
                   .agg(pl.col("err_volpts").mean().alias("mean_err_vp"),
                        pl.col("err_volpts").abs().mean().alias("mean_abs_err_vp"),
                        pl.len().alias("n_days"))
                   .sort("mean_abs_err_vp"))
        print(summ)
        print("[D] err is (model varswap - VIX) on the day's COMMON k-strip, vol points. "
              "A systematic NEGATIVE err is the truncation deficit; the cross-model ordering "
              "at identical truncation is the wing-quality ranking (a surplus signals "
              "spurious wing variance, not superior wings).")
    else:
        print("[D] VIX csv loaded but no pack date matched a VIX date.")
else:
    print(f"[D] gated: place a date,vix csv at {VIX_CSV} to activate.")

## Summary

This notebook converts the thesis's audit metrics into pricing objects, one per section:

- **A (local vol)**: the *repair rate* is the fraction of the surface unusable as a pricing
  model without intervention; the MC round-trip error measures dynamic self-consistency.
- **C (densities)**: butterfly violations restated as negative probability mass.
- **E (residual economics)**: IC + cost-aware decile spread on held-out, in-domain quotes;
  the λ=0 counterfactual prices arbitrage-dirtiness in signal quality.
- **F (trading)**: F1 shows executable arbitrage is (essentially) absent from the quotes;
  F2 sweeps the residual signal over holding horizon × execution assumption on a common,
  bucket-neutral, beyond-spread universe — gross and net, fit-gated.
- **G (detection power)**: the net-profitability frontier in (amplitude, persistence)
  space, calibrated on real spreads and measured noise, with the empirical residual
  distribution placed against it — and a built-in pipeline validation (a large injected
  signal must be detected).
- **D (VIX)**: external ground truth for the wings on the day's common k-strip.

**Production checklist:** (1) NB03 `NB03_LIMIT_DATES=None`, `NB03_EXPORT_ALL=1`,
`NB03_LAM0_STRIDE=1` (shard by `NB03_DATE_START/END` across terminals); (2) NB04
`NB04_RESUME=1`, `NB04_EXPORT_DAYS=386`; (3) re-run this notebook — every section scales
with whatever packs exist.

In [ ]:
# === THESIS_HEADLESS_BOOTSTRAP_V1 — figure manifest ===
write_figure_manifest()
